In [1]:
#April 22, 2026
#Charlotte Meyer
#Data Seminar
#This code is meant to use statsmodels to create a logistic regression

In [2]:
#importing packages
import statsmodels.api
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
import numpy as np
#import statsmodels.api as sm
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import train_test_split

In [3]:
#reading in the data 
cejst = pd.read_csv("cejst_logistic_regression.csv", index_col=0)

#setting pandas columns so I can see all of them
pd.set_option('display.max_columns', None)

cejst

/var/folders/sl/dyq7sj0x2_z0dr2x82v1ln_80000gn/T/ipykernel_30676/4195476803.py:2: DtypeWarning: Columns (0: Identified as disadvantaged, 1: Greater than or equal to the 90th percentile for expected agriculture loss rate and is low income?, 2: Tract experienced historic underinvestment and remains low income, 3: no_plumbing, 4: There is at least one abandoned mine in this census tract and the tract is low income., 5: Percent of the Census tract that is within Tribal areas) have mixed types. Specify dtype option on import or set low_memory=False.
  cejst = pd.read_csv("cejst_logistic_regression.csv", index_col=0)


,GEOID,County Name,State,perc_black,perc_native_american,perc_asian,perc_hawaiin_pacific,perc_mixed_race,perc_white,perc_hispanic,Percent other races,perc_children,Percent age 10 to 64,perc_elderly,Total threshold criteria exceeded,Total categories exceeded,Identified as disadvantaged without considering neighbors,Identified as disadvantaged based on neighbors and relaxed low income threshold only,Identified as disadvantaged due to tribal overlap,Identified as disadvantaged,perc_disadvantaged,Share of neighbors that are identified as disadvantaged,Identified as disadvantaged in v1.0,Identified as disadvantaged solely due to status in v1.0 (grandfathered),population,Interpolated number of off-campus students in poverty,Adjusted percent of individuals below 200% Federal Poverty Line (percentile),perc_poverty,Is low income?,Income data has been estimated based on geographic neighbor income,Greater than or equal to the 90th percentile for expected agriculture loss rate and is low income?,Expected agricultural loss rate (Natural Hazards Risk Index) (percentile),Expected agricultural loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected building loss rate and is low income?,Expected building loss rate (Natural Hazards Risk Index) (percentile),Expected building loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected population loss rate and is low income?,Expected population loss rate (Natural Hazards Risk Index) (percentile),loss_rate,Share of properties at risk of flood in 30 years (percentile),flood_risk,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years and is low income?,Share of properties at risk of fire in 30 years (percentile),Share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years and is low income?,Greater than or equal to the 90th percentile for energy burden and is low income?,Energy burden (percentile),energy_burden,Greater than or equal to the 90th percentile for PM2.5 exposure and is low income?,PM2.5 in the air (percentile),pm_air,Greater than or equal to the 90th percentile for diesel particulate matter and is low income?,Diesel particulate matter exposure (percentile),diesel_exposure,Greater than or equal to the 90th percentile for traffic proximity and is low income?,Traffic proximity and volume (percentile),traffic_prox,Greater than or equal to the 90th percentile for DOT transit barriers and is low income?,travel_barriers,Greater than or equal to the 90th percentile for housing burden and is low income?,Housing burden (percent) (percentile),housing_burden,"Greater than or equal to the 90th percentile for lead paint, the median house value is less than 90th percentile and is low income?",Percent pre-1960s housing (lead paint indicator) (percentile),Percent pre-1960s housing (lead paint indicator),Median value ($) of owner-occupied housing units (percentile),house_value,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent and is low income?,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent,Share of the tract's land area that is covered by impervious surface or cropland as a percent,pavement_cropland,Does the tract have at least 35 acres in it?,Tract experienced historic underinvestment and remains low income,Tract experienced historic underinvestment,no_plumbing,Share of homes with no kitchen or indoor plumbing (percent),Greater than or equal to the 90th percentile for proximity to hazardous waste facilities and is low income?,

In [6]:
#making a function that will split columns into bins for fixed effects

def split_into_bins(fixed_effect_column):

    #cut into 11 bins (arbitrary)
    #using cut, NOT qcut
    cejst[fixed_effect_column + "_bin"] = pd.cut(cejst[fixed_effect_column], 11, labels=range(1,12))
    
    #creating dummies, prefix adds column in front of bin name
    dummies = pd.get_dummies(cejst[fixed_effect_column + "_bin"])#, prefix=fixed_column_effect)
    
    #removing middle dummy
    dummies = dummies[[i for i in dummies.columns if i != 6]]
    
    #renaming dummies
    dummies.columns = [fixed_effect_column + "_" + str(i) for i in dummies.columns ]
    
    #combining existing columns with cejst
#    dummies[fixed_effect_column] = cejst[fixed_effect_column]
#    dummies['has_datacenter'] = cejst['has_datacenter']

    #drop na
    dummies = dummies.dropna()
    
    #viewing bins
    return dummies
#    return cejst.groupby('bin')[[fixed_effect_column, 'has_datacenter']].mean() #this shows the bins; will show bin 6

In [7]:
linguistic_isolation_dummies = split_into_bins("linguistic_isolation")
linguistic_isolation_dummies

cejst = cejst.copy()
cejst = pd.concat([cejst, linguistic_isolation_dummies], axis=1)

/var/folders/sl/dyq7sj0x2_z0dr2x82v1ln_80000gn/T/ipykernel_30676/23027534.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  cejst[fixed_effect_column + "_bin"] = pd.cut(cejst[fixed_effect_column], 11, labels=range(1,12))


In [8]:
perc_poverty_dummies = split_into_bins("perc_poverty")
perc_poverty_dummies

cejst = pd.concat([cejst, perc_poverty_dummies], axis=1)

In [10]:
cejst

,GEOID,County Name,State,perc_black,perc_native_american,perc_asian,perc_hawaiin_pacific,perc_mixed_race,perc_white,perc_hispanic,Percent other races,perc_children,Percent age 10 to 64,perc_elderly,Total threshold criteria exceeded,Total categories exceeded,Identified as disadvantaged without considering neighbors,Identified as disadvantaged based on neighbors and relaxed low income threshold only,Identified as disadvantaged due to tribal overlap,Identified as disadvantaged,perc_disadvantaged,Share of neighbors that are identified as disadvantaged,Identified as disadvantaged in v1.0,Identified as disadvantaged solely due to status in v1.0 (grandfathered),population,Interpolated number of off-campus students in poverty,Adjusted percent of individuals below 200% Federal Poverty Line (percentile),perc_poverty,Is low income?,Income data has been estimated based on geographic neighbor income,Greater than or equal to the 90th percentile for expected agriculture loss rate and is low income?,Expected agricultural loss rate (Natural Hazards Risk Index) (percentile),Expected agricultural loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected building loss rate and is low income?,Expected building loss rate (Natural Hazards Risk Index) (percentile),Expected building loss rate (Natural Hazards Risk Index),Greater than or equal to the 90th percentile for expected population loss rate and is low income?,Expected population loss rate (Natural Hazards Risk Index) (percentile),loss_rate,Share of properties at risk of flood in 30 years (percentile),flood_risk,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of flood in 30 years and is low income?,Share of properties at risk of fire in 30 years (percentile),Share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years,Greater than or equal to the 90th percentile for share of properties at risk of fire in 30 years and is low income?,Greater than or equal to the 90th percentile for energy burden and is low income?,Energy burden (percentile),energy_burden,Greater than or equal to the 90th percentile for PM2.5 exposure and is low income?,PM2.5 in the air (percentile),pm_air,Greater than or equal to the 90th percentile for diesel particulate matter and is low income?,Diesel particulate matter exposure (percentile),diesel_exposure,Greater than or equal to the 90th percentile for traffic proximity and is low income?,Traffic proximity and volume (percentile),traffic_prox,Greater than or equal to the 90th percentile for DOT transit barriers and is low income?,travel_barriers,Greater than or equal to the 90th percentile for housing burden and is low income?,Housing burden (percent) (percentile),housing_burden,"Greater than or equal to the 90th percentile for lead paint, the median house value is less than 90th percentile and is low income?",Percent pre-1960s housing (lead paint indicator) (percentile),Percent pre-1960s housing (lead paint indicator),Median value ($) of owner-occupied housing units (percentile),house_value,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent and is low income?,Greater than or equal to the 90th percentile for share of the tract's land area that is covered by impervious surface or cropland as a percent,Share of the tract's land area that is covered by impervious surface or cropland as a percent,pavement_cropland,Does the tract have at least 35 acres in it?,Tract experienced historic underinvestment and remains low income,Tract experienced historic underinvestment,no_plumbing,Share of homes with no kitchen or indoor plumbing (percent),Greater than or equal to the 90th percentile for proximity to hazardous waste facilities and is low income?,

In [11]:
# "'has_datacenter ~ ' + ' + '.join(predictors)"
#logistic regression model with fixed effects

# Define predictor variables
predictors = ['perc_white', 'perc_poverty', 'pm_air', 'perc_unemployment', 
              'hazard_waste_prox', 'life_expectancy', 'perc_hispanic', 
              'perc_elderly', 'perc_children', 'perc_disadvantaged', 
              'population', 'loss_rate', 'flood_risk', 'energy_burden', 
              'diesel_exposure', 'traffic_prox', 'travel_barriers', 
              'housing_burden', 'house_value', 'pavement_cropland', 
              'no_plumbing', 'superfund_prox', 'wastewater_discharge', 
              'asthma', 'diabetes', 'heart_disease', 'household_income_compared', 
              'linguistic_isolation', 'no_hs_degree']

#list of dummy dataframes
dummy_list = [linguistic_isolation_dummies, perc_poverty_dummies]
print(dummy_list)

#making list of dummy columns that predict a data center ??
for new_dummy in dummy_list:
    #dummies that predict data center
    dummy_predictors = [i for i in new_dummy.columns if i != 'has_datacenter']

#combining regular predictor columns with dummy predictor columns
all_predictors = predictors + dummy_predictors
print(all_predictors)

#fixed effects model
my_fixed_effects_model = statsmodels.formula.api.logit('has_datacenter ~ ' + ' + '.join(all_predictors), 
    data=cejst, subset=None, drop_cols=None).fit(disp=False) #, *args, **kwargs

my_fixed_effects_model.summary()


[       linguistic_isolation_1  linguistic_isolation_2  linguistic_isolation_3  \
0                        True                   False                   False   
1                        True                   False                   False   
2                        True                   False                   False   
3                       False                   False                   False   
4                       False                   False                   False   
...                       ...                     ...                     ...   
74129                   False                   False                   False   
74130                   False                   False                   False   
74131                   False                   False                   False   
74132                   False                   False                   False   
74133                   False                   False                   False   

       linguistic_isolatio

/Users/charlottemeyer/anaconda3/envs/.da_401/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:         has_datacenter   No. Observations:                44016
Model:                          Logit   Df Residuals:                    43976
Method:                           MLE   Df Model:                           39
Date:                Thu, 30 Apr 2026   Pseudo R-squ.:                 0.05066
Time:                        20:16:05   Log-Likelihood:                -1473.7
converged:                      False   LL-Null:                       -1552.3
Covariance Type:            nonrobust   LLR p-value:                 3.851e-16
=============================================================================================
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                    -2.9869      2.459     -1.215      0.225      -7.807       1.833
perc_poverty_1[T.True]        0.5902      1.124      0.525      0.599      -1.612       2.793
perc_poverty_2[T.True]       -0.0233      0.944     -0.025      0.980      -1.873       1.826
perc_poverty_3[T.True]        0.1413      0.736      0.192      0.848      -1.302       1.584
perc_poverty_4[T.True]        0.2081      0.531      0.392      0.695      -0.833       1.249
perc_poverty_5[T.True]       -0.7609      0.397     -1.916      0.055      -1.539       0.018
perc_poverty_7[T.True]        0.0440      0.403      0.109      0.913      -0.746       0.834
perc_poverty_8[T.True]       -0.1558      0.651     -0.239      0.811      -1.432       1.120
perc_poverty_9[T.True]       -0.7605      1.232     -0.617      0.537      -3.176       1.655
perc_poverty_10[T.True]     -13.2550   1147.040     -0.012      0.991   -2261.413    2234.903
perc_poverty_11[T.True]     -34.9017   2.65e+08  -1.32e-07      1.000   -5.19e+08    5.19e+08
perc_white                   -0.4491      0.555     -0.809      0.418      -1.537       0.639
perc_poverty                  0.2426      2.737      0.089      0.929      -5.121       5.606
pm_air                       -0.0089      0.048     -0.187      0.852      -0.102       0.084
perc_unemployment             0.0093      0.023      0.395      0.693      -0.037       0.055
hazard_waste_prox             0.0326      0.017      1.866      0.062      -0.002       0.067
life_expectancy               0.0112      0.024      0.464      0.643      -0.036       0.059
perc_hispanic                 0.1262      0.587      0.215      0.830      -1.025       1.277
perc_elderly                 -2.2831      1.905     -1.199      0.231      -6.016       1.450
perc_children                -0.7045      1.974     -0.357      0.721      -4.573       3.164
perc_disadvantaged            0.0032      0.003      1.202      0.229      -0.002       0.008
population                -8.732e-05   3.35e-05     -2.606      0.009      -0.000   -2.16e-05
loss_rate                    44.7383     51.387      0.871      0.384     -55.979     145.455
flood_risk                   -0.0025      0.005     -0.545      0.586      -0.012       0.007
energy_burden                -0.3828      0.084     -4.559      0.000      -0.547      -0.218
diesel_exposure              -0.2888      0.405     -0.713      0.476      -1.083       0.505
traffic_prox              -2.166e-06    3.8e-05     -0.057      0.955   -7.66e-05    7.23e-05
travel_barriers           -6.676e-05      0.003     -0.024      0.981      -0.006       0.005
housing_burden               -0.0138      0.011     -1.287      0.198      -0.035       0.007
house_value               -1.259e-06   4.77e-07     -2.639      0.008   -2.19e-06   -3.24e-07
pavement_cropland             0.0038      0.003      1.222      0.222      -0.002       0.010
no_plumbing                   0.0844      0.2

In [12]:
#list of columns that had low p-values in logit model
feature_columns_significant = ["perc_poverty",
                               "perc_hispanic",
                               "perc_elderly",
                               "population",
                               "loss_rate",
                               "travel_barriers",
                               "house_value",
                               "risk_facilities_prox",
                               "leaky_storage_tanks",
                               "asthma"]

new_significant = ["population", 
                   "energy_burden",
                   "risk_facilities_prox",
                   "linguistic_isolation"]
                   

In [13]:
X = cejst[new_significant]
y = cejst["has_datacenter"]

#dropping all rows with NaN in X columns
X = X.dropna()
#Keep y aligned with X
y = y.loc[X.index]  

# Verify they match
print(f"X shape: {X.shape}, y shape: {y.shape}")
#verify no na's
print(X.isnull().sum())

X shape: (72175, 4), y shape: (72175,)
population              0
energy_burden           0
risk_facilities_prox    0
linguistic_isolation    0
dtype: int64


In [14]:
#splittling into testing and training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=26
)

In [15]:
#code adapted from https://www.statology.org/how-to-test-for-multicollinearity-with-statsmodels/
#testing for multicollinearity

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Compute VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(X.shape[1])]

print(vif_data)

                feature       VIF
0            population  2.805788
1         energy_burden  1.201015
2  risk_facilities_prox  1.551619
3  linguistic_isolation  3.271948
